# Retrospective Tetanus Vaccination and Injury Records of Emergency Outpatients at Zhejiang Hospital Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.fvhy-dmxk/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Get metadata as dict
meta_obj = dataset.metadata
meta_dict = meta_obj.to_json()

# Print dataset title and description
print(f"{meta_dict['name']}: {meta_dict['description']}")

# Optionally, pretty print further context
pprint.pprint({'URL': meta_dict['@id'], 'Version': meta_dict['version'], 'Keywords': meta_dict.get('keywords', [])})

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, each entity (record set, field, etc.) is referenced by its `@id`. We'll inspect the schema to see what record sets are available and examine their structure.

In [ ]:
# Overview: record sets, fields, and columns
# Reference by @id. Find all record sets and their @ids.

record_sets = meta_dict.get('recordSet', [])
if not record_sets:
    print("No record sets listed in metadata. Trying dataset.record_sets...")
    record_sets = dataset.record_sets()
    if record_sets:
        print(f"Available record sets: {[rs['@id'] for rs in record_sets]}")
else:
    print(f"Record sets from metadata: {[rs.get('@id', rs) for rs in record_sets]}")

# If record sets are empty, try automatically discovering them
if not record_sets:
    # mlcroissant allows us to enumerate available record sets
    discovered_record_sets = dataset.record_sets()
    print("Discovered record sets:")
    for rs in discovered_record_sets:
        print(f"- {rs['@id']} (name: {rs.get('name', 'N/A')}, description: {rs.get('description', 'N/A')})")
else:
    # Print fields and their @ids for each record set
    for rs in record_sets:
        rs_id = rs.get('@id', rs) if isinstance(rs, dict) else rs
        print(f"Record Set @id: {rs_id}")
        # Try to get field info
        fields = rs.get('field', []) if isinstance(rs, dict) else []
        for f in fields:
            f_id = f.get('@id', f) if isinstance(f, dict) else f
            print(f"  Field @id: {f_id}")

# If record sets exist, show example records
if record_sets:
    # Reference by @id
    first_rs_id = record_sets[0].get('@id', record_sets[0]) if isinstance(record_sets[0], dict) else record_sets[0]
    print(f"\nSample records for Record Set '{first_rs_id}':")
    for record in dataset.records(record_set=first_rs_id):
        print(record)
        break  # Show only one for preview
else:
    print("No record sets found in metadata or via discovery. Data extraction will require manual exploration.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data: enumerate all discovered record sets
# Get all available record set @ids
record_sets = dataset.record_sets()
record_set_ids = [rs['@id'] for rs in record_sets]
print("Available Record Set @ids:")
for rs_id in record_set_ids:
    print(f" - {rs_id}")

# Load records from each record set into a DataFrame
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nDataFrame columns for record set '{record_set_id}':")
        print(df.columns.tolist())
        print(df.head())
    else:
        print(f"\nNo records found for record set '{record_set_id}'.")

# For demonstration, select the first available record set
if dataframes:
    preview_rs_id = list(dataframes.keys())[0]
    print(f"\nSelected Record Set for EDA: {preview_rs_id}")
else:
    preview_rs_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Use `@id` for referencing fields and columns. Demonstrate removing outliers, normalizing values, and grouping data.

**Note:** If variable names are not obvious, print available columns and choose a suitable numeric and grouping field.

In [ ]:
import numpy as np
from pandas.api.types import is_numeric_dtype

# Choose a record set for EDA
if preview_rs_id:
    df = dataframes[preview_rs_id]
    print("Columns available for EDA:")
    print(df.columns.tolist())

    # Try to auto-detect a numeric field
    numeric_fields = [col for col in df.columns if is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
    else:
        # Try to convert some text fields to numeric if possible
        numeric_field_id = df.columns[0]
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        numeric_fields = [numeric_field_id]

    print(f"Selected numeric field: {numeric_field_id}")

    # Filter: remove rows where numeric_field is not finite
    df_filtered = df[np.isfinite(df[numeric_field_id])]

    threshold = df_filtered[numeric_field_id].mean()  # Use mean as example threshold
    filtered_df = df_filtered[df_filtered[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Choose a group field: select next column, or if not available, skip
    group_field_candidates = [col for col in df.columns if col != numeric_field_id]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field} (mean of {numeric_field_id}):")
        print(grouped_df.head())
    else:
        print("No group field found for grouping. Skipping grouping step.")
else:
    print("No data available for EDA. Please check previous steps.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We will plot the distribution of the selected numeric field and show a boxplot grouped by the chosen group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if preview_rs_id and numeric_fields:
    # Histogram of numeric field
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot grouped by group_field
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"'{numeric_field_id}' by '{group_field}'")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No suitable numeric field or group field for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Croissant metadata provided rich context and schema details for the dataset.
- Record sets, fields, and columns were referenced by their `@id` for precision and reproducibility.
- Exploratory data analysis allowed initial filtering and normalization on numeric fields, and grouping by key attributes.
- Visualizations revealed the distribution and further grouped relationships within the dataset.

This notebook can be extended to apply machine learning models, imputation procedures, or domain-specific statistical tests on the FAIR^2 dataset. Refer to the Croissant schema for further detail and reproducibility by referencing entities using their `@id`.